# FGSM demo + ART evaluation

Apply the lightweight `robustness__simple_fgsm_perturbation` to a single feature vector, then attempt the full ART `robustness__evaluate_sklearn_robustness` (requires adversarial-robustness-toolbox).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import joblib, numpy as np
from examples.adversarial import swift_binary_dataset, train_test, train, fgsm_grad
from mcp_servers import robustness_tools as rt

X, y, feats, target = swift_binary_dataset(1000, seed=42)
Xtr, Xte, ytr, yte = train_test(X, y)
m = train('lr', Xtr, ytr)
joblib.dump(m, 'examples/adversarial/swift_lr.joblib')
np.save('examples/adversarial/X_test.npy', Xte)
np.save('examples/adversarial/y_test.npy', yte)
print('saved model + test arrays under examples/adversarial/')

In [ ]:
g = fgsm_grad(m, Xte[:1], yte[:1])[0]
print(rt.simple_fgsm_perturbation(Xte[0].tolist(), g.tolist(), eps=0.5))

In [ ]:
print(rt.evaluate_sklearn_robustness(
    'examples/adversarial/swift_lr.joblib',
    'examples/adversarial/X_test.npy',
    'examples/adversarial/y_test.npy',
    attack='ProjectedGradientDescent', eps=0.5, norm='inf'))